# Agent evolution and lineage

These records describe one evolutionary journal. Candidate identifiers describe historical artifacts. Some ancestry references point outside the observed candidate list. No candidate is ranked here by an unmeasured quantity.

In [ ]:
from pathlib import Path
import csv
import json
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "results/phase3/EXPERIMENT_MANIFEST.json").is_file())
results = root / "results"

def load_csv(relative):
    with (results / relative).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


In [ ]:
candidates = load_csv("evolution_candidates.csv")
by_generation = Counter(int(r["generation"]) for r in candidates)
nodes = {r["name"] for r in candidates}
links = [(p, r["name"]) for r in candidates for p in json.loads(r["parents_json"] or "[]")]
fig, ax = plt.subplots()
x = sorted(by_generation)
ax.bar(x, [by_generation[g] for g in x])
ax.set(xticks=x, xlabel="Recorded generation", ylabel="Candidates", title="Size of each observed generation")
plt.show()
print("Recorded parent links:", len(links))
print("Parents outside the candidate table:", len({p for p, _ in links if p not in nodes}))
print("Example recorded edges:", links[:5])

In [ ]:
matches = load_csv("evolution_matches.csv")
phases = sorted({r["phase"] for r in matches})
states = sorted({r["status"] for r in matches})
status = Counter((r["phase"], r["status"]) for r in matches)
base = [0] * len(phases)
fig, ax = plt.subplots()
for state in states:
    count = [status[p, state] for p in phases]
    ax.bar(phases, count, bottom=base, label=state)
    base = [a+b for a, b in zip(base, count)]
ax.set(ylabel="Match records", title="Outcomes and failed attempts by search phase")
ax.legend()
plt.show()